In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif

# ===================== 1. Load Dataset =====================
df = pd.read_csv("iris.csv")   # change filename if needed

print("===== ORIGINAL DATASET =====")
print(df.head(), "\n")

# ===================== 2. Missing Value Handling =====================
for col in df.columns:
    if df[col].dtype == "object":
        df[col].fillna(df[col].mode()[0], inplace=True)
    else:
        df[col].fillna(df[col].mean(), inplace=True)

# ===================== 3. Normalization =====================
numeric_cols = df.select_dtypes(include=np.number).columns
scaler = MinMaxScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# ===================== 4. Discretization =====================
col_to_bin = numeric_cols[0]   # first numeric column
df[col_to_bin + "_bin"] = pd.cut(df[col_to_bin], bins=3, labels=["Low", "Medium", "High"])

# ===================== 5. Attribute Selection =====================
X = df.drop(columns=[df.columns[-1]])  # features
y = df[df.columns[-1]]  # target

selector = SelectKBest(score_func=f_classif, k=2)
selector.fit(X.select_dtypes(include=np.number), y)

selected_features = X.select_dtypes(include=np.number).columns[selector.get_support()]

# Keep only selected attributes + target + discretized column
df_selected = df[selected_features.tolist() + [df.columns[-1], col_to_bin + "_bin"]]

# ===================== 6. Attribute Removal =====================
# Example: remove one attribute manually
if len(selected_features) > 1:
    df_selected = df_selected.drop(columns=[selected_features[0]])

# ===================== ✅ FINAL PREPROCESSED DATASET =====================
print("===== FINAL PREPROCESSED DATASET =====")
print(df_selected.head(20))   # print first 20 rows
print("\nShape of Final Dataset:", df_selected.shape)


===== ORIGINAL DATASET =====
   sepallength  sepalwidth  petallength  petalwidth           class
0         6.20        3.41         6.04        0.36  Iris-virginica
1         5.69        2.54         3.56        1.53  Iris-virginica
2         6.32        3.23         1.71        1.13     Iris-setosa
3         7.02        2.75         1.62        1.03     Iris-setosa
4         5.61        2.87         3.19        1.66     Iris-setosa 

===== FINAL PREPROCESSED DATASET =====
    sepalwidth sepallength_bin sepallength_bin
0     0.605042          Medium          Medium
1     0.361345          Medium          Medium
2     0.554622          Medium          Medium
3     0.420168            High            High
4     0.453782          Medium          Medium
5     0.495798          Medium          Medium
6     0.476190            High            High
7     0.302521          Medium          Medium
8     0.644258          Medium          Medium
9     0.490308          Medium          Medium
10   

/tmp/ipython-input-2496476573.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mean(), inplace=True)
/tmp/ipython-input-2496476573.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try usin